## Challenge 1: Byte Alchemist

File: challenge1.txt

Tugas:
- hex → bytes
- bytes → base64
- base64 → bytes
- bytes → int
- int → bytes
- pastikan hasil akhir identik

In [ ]:
import hashlib
from pathlib import Path
from base64 import b64encode, b64decode
from Crypto.Util.number import long_to_bytes

In [ ]:
# read the buffer
p = Path("/mnt/d/my-kisah/crypto/1.cryptopals/set_1/py_chall/")
f = p / "challenge1.txt"

# raw hex
h = f.read_text()

# hex -> bytes
bt = bytes.fromhex(h)

# bytes -> base64
b64 = b64encode(bt)

# base64 -> bytes
bb64 = b64decode(b64)

# bytes -> int
num = int.from_bytes(bb64, "big")

# int -> bytes
byte_num = long_to_bytes(num)

# turn byte_num into digest sha256
m = hashlib.sha256()
m.update(byte_num)
ct = m.hexdigest()

# print-out
print(f"hex form           :   {h}")
print(f"bytes form         :   {bt}")
print(f"base64 form        :   {b64}")
print(f"bytes form         :   {bb64}")
print(f"integer form       :   {num}")
print(f"byte integer form  :   {byte_num}")
print(f"sha256 form        :   {ct}")

### Validation

In [ ]:
answer = ct
sha256 = "02f9d39e787d2dd2536beeb8b1711cc92b58d3a7ab9fa84f0af718cce6f7d0ff"

if answer == sha256:
    print("correct...")
else:
    print("try again...")

## Challenge 2: XOR Twins

Referensi: Sesi 2 (Fixed XOR) + Sesi 3 (Single-byte XOR cipher)

Tugas:

1. Buat fungsi fixed_xor(buf1: bytes, buf2: bytes) -> bytes — XOR dua buffer panjang sama, byte-per-byte. Kalau panjangnya beda, raise ValueError.
2. Buat fungsi single_byte_xor(data: bytes, key: int) -> bytes — XOR seluruh data dengan satu key byte (0–255) berulang.
3. Buat fungsi scorer score_english(text: bytes) -> float — pakai frequency analysis (ETAOIN SHRDLU) buat nilai seberapa "mirip English" hasil decode-nya. Bebas metode: character frequency table, atau itung rasio printable ASCII, dsb.
4. Buat fungsi crack_single_byte_xor(ciphertext: bytes) -> tuple[int, bytes, float] — brute-force 256 key, return (key, plaintext, score) terbaik.

In [ ]:
def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))
    
def single_byte_xor(data: bytes, key: int):
    return bytes(bytearray([d ^ key for d in data]))

def score_english(text: bytes):
    ascii = [i for i in range(32,127)]
    letters = [69, 84, 65, 79, 73, 78, 83, 72, 82, 68, 76, 67, 85, 101, 116, 97, 111, 105, 110, 115, 104, 114, 100, 108, 99, 117, 32]
    penalty = [mins for mins in b"!@#$%^&*"]
    score = 0

    for t in text:
        if t not in ascii:
            return 0
        if t in letters:
            match t:
                case 69 | 97:
                    score += 5
                case 84 | 111:
                    score += 3
                case 65 | 105:
                    score += 2
                case 32 | 110:
                    score += 4
            score += 1
        if t in penalty:
            score -= 1
    return score/len(text)

def crack_single_byte_xor(ciphertext: bytes):
    cand = []

    for k in range(256):
        p = b""
        p += single_byte_xor(ciphertext, k)

        score = score_english(p)
        if k == 0:
            cand.append(ciphertext)
            cand.append(k)
            cand.append(p)
            cand.append(score)
            continue

        if score < cand[3]:
            continue
        if b" " not in p or b"  " in p:
            continue
        else:
            cand[0] = ciphertext.hex()
            cand[1] = k
            cand[2] = p
            cand[3] = score
    
    return tuple(cand)

In [ ]:
ciphertext_hex = "1b37373331363f78151b7f2b783431333d78397828372d363c78373e783a393b3736"
ct = bytes.fromhex(ciphertext_hex)

cand = crack_single_byte_xor(ct)
key = cand[0]
pt = cand[1]
score = cand[2]

print(f"ciphertext  : {ct}")
print(f"key         : {key}")
print(f"plaintext   : {pt.decode()}")
print(f"score       : {score}")

print("\nTesting Self-XOR:")
zeroed = fixed_xor(pt, pt)
assert zeroed == b"\x00" * len(pt), "fixed_xor lo salah, bro"
print(zeroed)  # harus print b'\x00\x00\x00...\x00' sepanjang plaintext

## Challenge 3: The Needle in the Haystack


Referensi: Sesi 4 (Detect single-character XOR)

Tugas:
1. Reuse crack_single_byte_xor dari Challenge 2.
2. Kamu akan dikasih list of hex strings (banyak baris, cuma 1 yang beneran hasil single-byte XOR encrypt dari English plaintext).
3. Buat fungsi find_xor_encrypted_line(candidates: list[str]) -> tuple[int, int, bytes, float] yang:
- Decode tiap baris dari hex ke bytes
- Jalanin crack_single_byte_xor ke tiap baris
- Return index baris + hasil crack terbaik: (line_index, key, plaintext, score)



In [ ]:
def single_byte_xor(data: bytes, key: int):
    return bytes(bytearray([d ^ key for d in data]))

def score_english(text: bytes):
    ascii = [i for i in range(32,127)]
    letters = [69, 84, 65, 79, 73, 78, 83, 72, 82, 68, 76, 67, 85, 101, 116, 97, 111, 105, 110, 115, 104, 114, 100, 108, 99, 117, 32]
    penalty = [mins for mins in b"!@#$%^&*"]
    score = 0

    for t in text:
        if t not in ascii:
            return 0
        if t in letters:
            match t:
                case 69 | 97:
                    score += 5
                case 84 | 111:
                    score += 3
                case 65 | 105:
                    score += 2
                case 32 | 110:
                    score += 4
            score += 1
        if t in penalty:
            score -= 1
    return score/len(text)

def crack_single_byte_xor(ciphertext: bytes):
    cand = []

    for k in range(256):
        p = b""
        p += single_byte_xor(ciphertext, k)

        score = score_english(p)
        if k == 0:
            cand.append(ciphertext)
            cand.append(k)
            cand.append(p)
            cand.append(score)
            continue

        if score < cand[3]:
            continue
        if b" " not in p or b"  " in p:
            continue
        else:
            cand[0] = ciphertext.hex()
            cand[1] = k
            cand[2] = p
            cand[3] = score
    
    return tuple(cand)

In [ ]:
lines = [
    "ff744733478c3a0aabf919f21d20b31aa5008dfa610d23dab5c77928f168a930",
    # "0d313c79282c303a32793b2b362e37793f362179332c34292a79362f3c2b792d313c7935382320793d363e",
    "5368652073656c6c73207365617368656c6c73206279207468652073656173686f726520746f646179",
    "8d3d595751d0e94ae648f88680ed469da9837da2801c59c4047354f7ea59bd744c8b",
    "c1153ee1d421be3e787af0a74d56a60ddb62194c90e8baef495b",
    "84eae9f942a27844aa6d1816c46bd691c5c2924c57c553060dd070ba9b0f2920ca775df7b3",
    "2cf7e0877d5a557c4d7e795ccf5e673e3c83e5c6287a6f5f5d29a2536b0552e5b8345962d4dd88b6364678",
    "6f4e91356ba7c1cee08ebc8746da09a3fe349448f44ec5b9efed8967a683a5e403050102422ae596d8100dfc",
    "2030233030202330303023",
    "c90b5868bfd8d7cc5dd11cbf65a4f827894196485a98c4f56a5f566262fc5eb016",
]

best = None
for line in lines:
    try:
        ct = bytes.fromhex(line)
    except ValueError:
        ct = line.encode()
    
    cand = crack_single_byte_xor(ct)
    if best is None or cand[3] > best[3]:
        best = cand

ct, key, pt, score = best

print(f"ciphertext  : {ct}")
print(f"key         : {key}")
print(f"plaintext   : {pt}")
print(f"score       : {score}")

## Challenge 4: The Weaver


Referensi: Sesi 5 (Implement repeating-key XOR)

Tugas:

1. Buat fungsi repeating_key_xor(data: bytes, key: bytes) -> bytes
2. key diaplikasikan berulang (cycling) ke tiap byte data. Byte pertama data di-XOR sama byte pertama key, byte kedua data sama byte kedua key, dst. Kalau data lebih panjang dari key, key-nya "muter lagi" dari awal.

In [ ]:
def repeating_key_xor(data: bytes, key: bytes) -> bytes:
    return bytes(bytearray([key[i%len(key)] ^ d for i, d in enumerate(data)]))

### Test Case

Cek: repeating_key_xor(plaintext, key).hex() == expected_hex harus True.

#### Twist kecil (biar gak sekadar nyalin):

Buktikan bahwa repeating_key_xor itu self-inverse — artinya kalau lo XOR hasil encrypt-nya pakai key yang sama, lo harus dapet plaintext asli lagi:

```
encrypted = repeating_key_xor(plaintext, key)
decrypted = repeating_key_xor(encrypted, key)
assert decrypted == plaintext
```

In [ ]:
plaintext = b"Burning 'em, if you ain't quick and nimble\nI go crazy when I hear a cymbal"
key = b"ICE"

ct = repeating_key_xor(plaintext, key)
pt = ""
expected_hex = "0b3637272a2b2e63622c2e69692a23693a2a3c6324202d623d63343c2a26226324272765272a282b2f20430a652e2c652a3124333a653e2b2027630c692b20283165286326302e27282f"

print("process encrypt:")
print(f"expected cipher  : {expected_hex}")
print(f"output cipher    : {ct.hex()}")
print(f"result           : {ct.hex() == expected_hex}")

try:
    assert ct.hex() == expected_hex
    pt = repeating_key_xor(ct, key)
except AssertionError:
    print(f"message          : expected cipher does not match the output cipher")

print("\nreverse hasil encrypt:")
print(f"expected plain text      : {plaintext}")
print(f"output plain text        : {pt}")
print(f"result                   : {plaintext == pt}")

try:
    assert pt == plaintext
except AssertionError:
    print(f"message                  : expected plain text does not match the output plain text")

## Challenge 5: The Vigenère Breaker


### Referensi: Sesi 6 (Break repeating-key XOR)

Kita pecah jadi beberapa sub-step biar gak overwhelming, tapi tetap 1 challenge:

### Step 1 — Hamming Distance
Buat fungsi hamming_distance(b1: bytes, b2: bytes) -> int yang ngitung jumlah bit yang beda antara dua buffer sama panjang.

In [ ]:
def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))

def hamming_distance(b1: bytes, b2: bytes):
    xord = fixed_xor(b1, b2)
    return int.from_bytes(xord,"big").bit_count()

In [ ]:
try:
    assert hamming_distance(b"this is a test", b"wokka wokka!!!") == 37
    print("test case success!!!")
except AssertionError:
    print("test case failed. try to check the hamming functions...")

### Step 2 — Guess Key Size

Buat fungsi guess_keysize(ciphertext: bytes, min_size=2, max_size=40) -> list[int] yang:

1. Untuk tiap keysize dari min_size sampai max_size:

- Ambil beberapa chunk berturut-turut sepanjang keysize dari ciphertext (minimal 2 chunk, makin banyak makin akurat — misal 4 chunk)
- Hitung hamming distance rata-rata antar chunk-chunk itu
- Normalisasi: bagi hasilnya dengan keysize


2. Return list keysize yang diurutkan dari normalized distance terkecil (kandidat paling mungkin duluan)

In [ ]:
def repeating_key_xor(data: bytes, key: bytes) -> bytes:
    return bytes(bytearray([key[i%len(key)] ^ d for i, d in enumerate(data)]))

def guess_keysize(ciphertext: bytes, min_size = 2, max_size = 40):
    result = []

    for ks in range(min_size, max_size):
        hamming = 0
        pairs = 0
        chunks = [ciphertext[i*ks:(i + 1)*ks] for i in range(len(ciphertext))]
        # print(chunks[:2])
        for i, c in enumerate(chunks):
            if len(c) != ks or len(chunks[i+1]) != ks:
                continue
            score = hamming_distance(c, chunks[i+1])
            hamming += score
            pairs += 1
        
        if pairs == 0:
            continue
        avg_normalized = (hamming / pairs) / ks 
        result.append((ks, avg_normalized))
    
    result.sort(key=lambda x: x[1])
    return [ks for ks, _ in result]


In [ ]:
pt = b"aku suka anak tk aku suka anak tk aku suka anak tk aku suka anak tk aku suka anak tk aku suka anak tk aku suka anak tk aku suka anak tk aku suka anak tk aku suka anak tk"
key = b"sus banget loh ya"

ct = repeating_key_xor(pt, key)
print(ct)
ks = guess_keysize(ct)
print(ks)
print(len(key))

### Step 3: Transpose & Crack


#### Tugas:
Buat fungsi break_repeating_key_xor(ciphertext: bytes, keysize: int) -> bytes yang mengembalikan key yang ditemukan (bukan plaintext-nya dulu — key dulu, plaintext bisa didapat belakangan pakai repeating_key_xor).

Logikanya, langkah demi langkah:

1. Pecah ciphertext jadi blok-blok sepanjang keysize. (Ini beda sama chunking di Step 2 — sekarang kita pecah seluruh ciphertext, bukan cuma ambil beberapa chunk buat sampling.)
2. Transpose. Ini bagian intinya. Bikin keysize buah "blok baru", di mana:
- Blok baru ke-0 = kumpulan byte ke-0 dari setiap blok asli
- Blok baru ke-1 = kumpulan byte ke-1 dari setiap blok asli
- dst, sampai blok baru ke-(keysize-1)

Kenapa ini works: karena repeating_key_xor itu ngulang key secara siklik, byte ke-0 dari semua blok asli itu di-XOR pakai key byte yang sama (key byte pertama). Jadi kalau kita kumpulin semua byte itu jadi 1 "blok baru", itu sebenernya adalah single-byte XOR ciphertext — persis kayak Challenge 2/3!

3. Crack tiap blok hasil transpose pakai crack_single_byte_xor yang udah lo punya. Tiap blok bakal ngasih 1 key byte.
4. Gabungin semua key byte itu (sesuai urutan blok transpose-nya) → itu key final.

In [ ]:
def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))

def hamming_distance(b1: bytes, b2: bytes):
    xord = fixed_xor(b1, b2)
    return int.from_bytes(xord,"big").bit_count()

In [ ]:
def repeating_key_xor(data: bytes, key: bytes) -> bytes:
    return bytes(bytearray([key[i%len(key)] ^ d for i, d in enumerate(data)]))

def guess_keysize(ciphertext: bytes, min_size = 2, max_size = 40):
    result = []

    for ks in range(min_size, max_size):
        hamming = 0
        pairs = 0
        chunks = [
            ciphertext[i:i+ks] 
            for i in range(0,len(ciphertext),ks)
            ][:8]
        
        scores = []
        
        for i in range(len(chunks) - 1):
            for j in range(i+1, len(chunks)):
                if len(chunks[i]) != ks or len(chunks[j]) != ks:
                    continue

                scores.append(
                    hamming_distance(
                        chunks[i], chunks[j]
                        ) / ks
                )
        if not scores:
            continue
        avg_normalized = sum(scores)/len(scores)
        result.append((ks, avg_normalized))
    
    result.sort(key=lambda x: x[1])
    return [ks for ks, _ in result]

In [ ]:
def single_byte_xor(data: bytes, key: int):
    return bytes(bytearray([d ^ key for d in data]))

def score_english(text: bytes):
    score = 0

    freq = b"etaoinshrdlu ETAOINSHRDLU"

    for c in text:

        # printable + newline/tab
        if c in b"\n\r\t":
            score += 1

        elif c < 32 or c > 126:
            score -= 20
            continue

        if c in freq:
            score += 3

        if c == ord(" "):
            score += 5

        if chr(c).isalpha():
            score += 1

        if c in b"~@#$%^&*{}[]|":
            score -= 2

    return score

def crack_single_byte_xor(ciphertext: bytes):
    best = None

    for k in range(256):
        p = single_byte_xor(ciphertext,k)
        score = score_english(p)

        if best is None or score > best[3]:
            best = (
                ciphertext.hex(),
                k,
                p,
                score
            )

    return best

In [ ]:
def break_repeating_key_xor(ciphertext: bytes, keysize: int):
    blocks = [ciphertext[i:i+keysize] for i in range(0,len(ciphertext),keysize)]
    transpose = []

    for i in range(keysize):
        tmp = []
        for block in blocks:
            if i < len(block):
                tmp.append(block[i])
        transpose.append(bytes(tmp))

    key = b""
    for k in transpose:
        cand = crack_single_byte_xor(k)
        key += bytes([cand[1]])
    return key

In [ ]:
ct_hex = "25091506412c4c15040448150a4c14041504080b0b52091509171c4c1f0404090d452552061f1c41044c14141c0441110d1c0a50090f014c1341131d130c0307125005080b085e413948060a18520702010211051d0f50010f45010b411506060c021741110605452552051f061545071c0e0748160d155c41240004451f0600021b41041e174111040802021705501c0e0b0515090448120a4c3b41020105004052061f060f044c1109111b0445181a04501b140b1f1715501d0f11051e413948070c02164107000417095215180d4117031305500c08160d0211150913164c1b0f040741110417410303184b4c210e1d0d41010d0b12501c09004c01081c0d0f060952151509020d0901411d0713004c060911064104020b4112070e0e4c1d0f501c09004c0109150407494c130f14480413090018501f130a021541041d130b4c0600050f09114c1f04501b0e08090609190606450217165009030a1906411d11120000144f"
ct = bytes.fromhex(ct_hex)

ks = guess_keysize(ct)
keys = []

for s in ks[:3]:
    cand_key = break_repeating_key_xor(ct, s)
    keys.append(cand_key)

for k in keys:
    cand_pt = repeating_key_xor(ct, k)
    print(cand_pt)
    break

## Challenge 6: The Steel Box
### Referensi: Sesi 7 (AES in ECB mode)

Ini beda banget nuansanya dari Challenge 1-5 — kalau XOR itu "cipher yang bisa dibedah manual," AES itu block cipher modern yang gak bisa (dan gak perlu) lo implementasi dari nol. Fokus challenge ini murni di penggunaan library crypto yang bener, karena skill ini bakal lo pake terus di CTF/real-world (pentest-agent VPS lo, misalnya).

#### Tugas:

1. Install pycryptodome (pip install pycryptodome --break-system-packages kalau lo test di environment lo).
2. Buat fungsi aes_ecb_decrypt(ciphertext: bytes, key: bytes) -> bytes yang decrypt AES-128-ECB.
3. Key-nya: b"YELLOW SUBMARINE" (persis 16 byte, cek dulu emang 16 sebelum dipake — AES-128 wajib key 16 byte).
4. Ciphertext-nya base64-encoded di file asli cryptopals (https://cryptopals.com/static/challenge-data/7.txt)

#### Twist: 
Sebelum decrypt, cek dulu: ECB mode itu stateless & deterministic (2 blok plaintext yang identik selalu ngehasilin 2 blok ciphertext yang identik). Setelah lo decrypt file itu, coba encrypt ulang hasil plaintext-nya (pake AES.new(key, AES.MODE_ECB) lagi, mode encrypt), terus cocokin sama ciphertext asli byte-per-byte. Ini validasi bahwa encrypt/decrypt lo simetris dengan benar — mirip prinsip self-inverse yang kita pake di Challenge 4.

In [ ]:
!pip install pycryptodome

In [ ]:
import base64
from Crypto.Cipher import AES

In [ ]:
def aes_ecb_decrypt(ciphertext: bytes, key: bytes):
    ct = AES.new(key, AES.MODE_ECB)
    pt = ct.decrypt(ciphertext)
    return pt

def aes_ecb_encrypt(plaintext: bytes, key: bytes):
    pt = AES.new(key, AES.MODE_ECB)
    ct = pt.encrypt(plaintext)
    return ct

In [ ]:
with open("/mnt/d/my-kisah/crypto/1.cryptopals/files/7.txt") as f:
    b64_data = "".join(
        line.strip()
        for line in f
    )

cipher = base64.b64decode(b64_data)
key = b"YELLOW SUBMARINE"
plain = aes_ecb_decrypt(cipher, key)
# print("".join(sentc for sentc in plain.decode()))

print(cipher)
ct = aes_ecb_encrypt(plain, key)
print(ct)

## Challenge 7: The Fingerprint
### Referensi: Sesi 8 (Detect AES in ECB mode)


### Tugas:
Buat fungsi detect_ecb(ciphertexts: list[bytes], block_size=16) -> int yang:

1. Pecah tiap ciphertext jadi blok-blok 16 byte.
2. Cek: apakah ada blok yang berulang (identik) di dalam ciphertext yang sama?
3. Ciphertext dengan blok berulang paling banyak → itu kemungkinan besar dienkripsi pakai ECB.
4. Return index dari ciphertext itu di dalam list.

In [226]:
def split_block(ciphertext: bytes, size = 16):
    blocks = []
    for i in range(0, len(ciphertext), size):
        blocks.append(ciphertext[i:i+size])
    return blocks

def check_repeated(block: list) -> int:
    block_length = len(block)
    uniq_block = len(set(block))
    diff = abs(block_length - uniq_block)
    if diff != 0:
        return diff
    return 0

In [ ]:
with open("/mnt/d/my-kisah/crypto/1.cryptopals/files/8.txt") as f:
    h = f.readlines()

for idx, c in h:
    ct = bytes.fromhex(c.strip())
    blocks = split_block(ct)
    repeated = check_repeated(blocks)
    if repeated > 0:
        print(f"Ciphertext      : {ct}")
        print(f"Hex Ciphertext  : {c.strip()}")
        print(f"Index ke        : {idx}")

Ciphertext      : b'\xd8\x80a\x97@\xa8\xa1\x9bx@\xa8\xa3\x1c\x81\n=\x08d\x9a\xf7\r\xc0oO\xd5\xd2\xd6\x9ctL\xd2\x83\xe2\xdd\x05/kd\x1d\xbf\x9d\x11\xb04\x85B\xbbW\x08d\x9a\xf7\r\xc0oO\xd5\xd2\xd6\x9ctL\xd2\x83\x94u\xc9\xdf\xdb\xc1\xd4e\x97\x94\x9d\x9c~\x82\xbfZ\x08d\x9a\xf7\r\xc0oO\xd5\xd2\xd6\x9ctL\xd2\x83\x97\xa9>\xab\x8dj\xec\xd5fH\x91Tx\x9ak\x03\x08d\x9a\xf7\r\xc0oO\xd5\xd2\xd6\x9ctL\xd2\x83\xd4\x03\x18\x0c\x98\xc8\xf6\xdb\x1f*?\x9c@@\xde\xb0\xabQ\xb2\x993\xf2\xc1#\xc5\x83\x86\xb0o\xba\x18j'
Hex Ciphertext  : d880619740a8a19b7840a8a31c810a3d08649af70dc06f4fd5d2d69c744cd283e2dd052f6b641dbf9d11b0348542bb5708649af70dc06f4fd5d2d69c744cd2839475c9dfdbc1d46597949d9c7e82bf5a08649af70dc06f4fd5d2d69c744cd28397a93eab8d6aecd566489154789a6b0308649af70dc06f4fd5d2d69c744cd283d403180c98c8f6db1f2a3f9c4040deb0ab51b29933f2c123c58386b06fba186a
Index ke        : 132
